In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [2]:
df_main = pd.read_csv("~/mission_sih/mission_sih/data/nasa_data_cleaned.csv")


In [3]:

df_cat = pd.read_csv("~/mission_sih/mission_sih/data/land_use_data.csv")

In [4]:
df_main['acq_date'] = pd.to_datetime(df_main['acq_date'])
df_main['year'] = df_main['acq_date'].dt.year

In [5]:

EARTH_RADIUS_M = 6371000
RADIUS_LIMIT_M = 2000

In [6]:
df_cat.info()

<class 'pandas.DataFrame'>
RangeIndex: 17650701 entries, 0 to 17650700
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   year       int64  
 1   land_type  str    
 2   latitude   float64
 3   longitude  float64
dtypes: float64(2), int64(1), str(1)
memory usage: 538.7 MB


In [7]:
df_main['category'] = pd.Series([np.nan] * len(df_main), dtype='object')
df_main['match_dist_m'] = np.nan

In [8]:
for yr, group in df_main.groupby('year'):
    cat_group = df_cat[df_cat['year'] == yr]
    if cat_group.empty:
        continue  # no category data for this year, leave unmatched

    cat_rad = np.radians(cat_group[['latitude', 'longitude']].values)
    main_rad = np.radians(group[['latitude', 'longitude']].values)

    nn = NearestNeighbors(n_neighbors=1, metric='haversine', n_jobs=-1)
    nn.fit(cat_rad)
    dist, idx = nn.kneighbors(main_rad)

    dist_m = dist.flatten() * EARTH_RADIUS_M
    matched_cat = cat_group.iloc[idx.flatten()]['land_type'].values.astype(object)

    # apply 1km cutoff
    matched_cat = np.where(dist_m <= RADIUS_LIMIT_M, matched_cat, np.nan)

    df_main.loc[group.index, 'category'] = matched_cat
    df_main.loc[group.index, 'match_dist_m'] = dist_m

In [9]:
df_cat2 = pd.read_csv("~/mission_sih/mission_sih/data/osm_data_cleaned.csv")

In [10]:
# df_cat2 has lat, lon, category — no year column, direct global search
cat2_rad = np.radians(df_cat2[['latitude', 'longitude']].values)
main_rad = np.radians(df_main[['latitude', 'longitude']].values)

nn2 = NearestNeighbors(n_neighbors=1, metric='haversine', n_jobs=-1)
nn2.fit(cat2_rad)
dist2, idx2 = nn2.kneighbors(main_rad)

dist2_m = dist2.flatten() * EARTH_RADIUS_M
matched_cat2 = df_cat2.iloc[idx2.flatten()]['osm_category'].values.astype(object)

RADIUS_LIMIT_M_2 = 2000

# candidate is valid only if within 2km
valid2 = dist2_m <= RADIUS_LIMIT_M_2

# overwrite only if: valid AND (no previous match OR this one is closer)
prev_dist = df_main['match_dist_m'].values
no_prev_match = df_main['category'].isna().values
closer = dist2_m < prev_dist

should_replace = valid2 & (no_prev_match | closer)

df_main.loc[should_replace, 'category'] = matched_cat2[should_replace]
df_main.loc[should_replace, 'match_dist_m'] = dist2_m[should_replace]

In [14]:
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 5089539 entries, 0 to 5089538
Data columns (total 15 columns):
 #   Column        Dtype         
---  ------        -----         
 0   latitude      float64       
 1   longitude     float64       
 2   brightness    float64       
 3   scan          float64       
 4   track         float64       
 5   acq_date      datetime64[us]
 6   acq_time      int64         
 7   confidence    int64         
 8   bright_t31    float64       
 9   frp           float64       
 10  daynight      int64         
 11  type          int64         
 12  year          int32         
 13  category      object        
 14  match_dist_m  float64       
dtypes: datetime64[us](1), float64(8), int32(1), int64(4), object(1)
memory usage: 563.0+ MB


In [ ]:
df_main = df_main.dropna(subset=["category"])
df_main.reset_index(inplace=True)

In [26]:
df_main.drop(columns=["index"],inplace=True)

In [38]:
df = df_main.copy()

In [ ]:
df["track_scan"] = df["scan"]*df["track"]

In [40]:
df["confidence"] = df["confidence"].map({0:1, 1:2, 2:3})

In [41]:
df["final_bright"] = df["brightness"]*df["bright_t31"]
df["radiation"] = df["track_scan"]*df["frp"]
df["confidence"] = df["confidence"]*300

In [61]:
popped_column = df.pop("match_dist_m")

df.insert(14, "match_dist_m", popped_column)


In [62]:
df.head()

,latitude,longitude,scan,track,track_scan,frp,radiation,brightness,bright_t31,final_bright,acq_time,daynight,confidence,type,match_dist_m,category
0,23.04102,92.52061,0.62,0.71,0.4402,6.39,2.812878,336.69,296.72,99902.6568,606,0,600,0,307.030507,Forest
1,22.50259,92.55136,0.63,0.72,0.4536,8.80,3.991680,330.26,299.59,98942.5934,606,0,600,0,313.398197,Forest
2,22.50408,92.55746,0.63,0.72,0.4536,10.10,4.581360,346.15,298.87,103453.8505,606,0,600,0,548.842875,Forest
3,22.41149,92.65405,0.63,0.72,0.4536,3.30,1.496880,330.42,300.69,99353.9898,606,0,300,0,102.962301,Forest
4,22.12835,92.67033,0.63,0.72,0.4536,30.87,14.002632,330.93,297.33,98395.4169,606,0,600,0,421.911597,Forest


In [63]:
df.to_csv("~/mission_sih/mission_sih/data/final_dataset.csv", index=False)